<a href="https://colab.research.google.com/github/Hania-Emaan/urdu-ocr-codesaviours-si26-Hania-/blob/main/SI26_Week4_Hania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook loads the pretrained TrOCR model, fine-tunes it on our Urdu OCR dataset,
evaluates its accuracy on unseen test images, and saves the trained model to Google Drive.

In [1]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')

from transformers import RobertaTokenizer, ViTImageProcessor, VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

Mounted at /content/drive
Using device: cuda


tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
# Cell 2: Add Urdu tokens BEFORE moving model to device or setting config
urdu_letters = list("ابپتٹثجچحخدڈذرڑزژسشصضطظعغفقکگلمنںوہھءیےآأؤئة")
urdu_digits = list("۰۱۲۳۴۵۶۷۸۹")
urdu_punct = list("۔؟،؛٫٪ًٌٍَُِّْٰٓ")
new_tokens = list(set(urdu_letters + urdu_digits + urdu_punct))

num_added = tokenizer.add_tokens(new_tokens)
print(f'Added {num_added} new tokens')

# Resize and explicitly retie weights
model.decoder.resize_token_embeddings(len(tokenizer))
model.decoder.tie_weights()

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = len(tokenizer)
model.config.decoder.vocab_size = len(tokenizer)

model = model.to(device)
print('Model configured and moved to device')

Added 70 new tokens


[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model configured and moved to device


In [3]:
# Cell 3: Dataset
import os, pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, img_root, tokenizer, image_processor):
        self.data = pd.read_csv(csv_path, encoding='utf-8')
        self.img_root = img_root
        self.tokenizer = tokenizer
        self.image_processor = image_processor
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(os.path.join(self.img_root, row['image'])).convert('RGB')
        encoding = self.image_processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.tokenizer(row['text'], padding='max_length', max_length=128, truncation=True).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

raw_base = '/content/drive/MyDrive/urdu-ocr-si26/data/raw'
csv_path = '/content/drive/MyDrive/urdu-ocr-si26/data/labels.csv'
dataset = UrduOCRDataset(csv_path, raw_base, tokenizer, image_processor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
print(f'Training: {train_size}, Testing: {test_size}')

Training: 167, Testing: 42


In [4]:
# Cell 4: Train
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)
optimizer = AdamW(model.parameters(), lr=3e-5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch + 1} complete | Average Loss: {total_loss/len(train_loader):.4f}')
print('\nTraining complete!')


Epoch 1/3
  Batch 0/42 | Loss: 17.3123
  Batch 10/42 | Loss: 4.5638
  Batch 20/42 | Loss: 3.5516
  Batch 30/42 | Loss: 1.3608
  Batch 40/42 | Loss: 1.9482
Epoch 1 complete | Average Loss: 3.6286

Epoch 2/3
  Batch 0/42 | Loss: 2.6364
  Batch 10/42 | Loss: 2.3761
  Batch 20/42 | Loss: 1.7334
  Batch 30/42 | Loss: 1.8906
  Batch 40/42 | Loss: 1.6445
Epoch 2 complete | Average Loss: 1.8352

Epoch 3/3
  Batch 0/42 | Loss: 1.0785
  Batch 10/42 | Loss: 1.3330
  Batch 20/42 | Loss: 1.3282
  Batch 30/42 | Loss: 1.0069
  Batch 40/42 | Loss: 1.0479
Epoch 3 complete | Average Loss: 1.2396

Training complete!


In [5]:
model.eval()
print('=== Model Evaluation on Test Images ===\n')

correct = 0
total = 0
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        # CHANGED: added num_beams and repetition_penalty
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64,
            num_beams=4,
            repetition_penalty=2.0
        )
        generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted:  
Actual: سندھ ہائیکورٹ نے رجب بٹ کے خلاف توہین مذہب کیس کا فیصلہ سنا دیا

Predicted: 
Actual: دل میں ایک کانٹا سا چبھا۔

Predicted: 
Actual: جہاں نما۔

Predicted: 
Actual: ڈاکٹر عرفان

Predicted: 
Actual: ٹھنڈ میں ایک قحط زدہ گاؤں

Predicted: 
Actual: اینڈ اسٹیمپ میکر

Predicted: 
Actual: وفاقی اردو یونیورسٹی

Predicted: 
Actual: فٹبال دنیا بھر میں مقبول کھیل ہے۔

Predicted:  
Actual: آج کے دور میں سائنس اور ٹیکنالوجی کی ضرورت ہے

Predicted: 
Actual: بانگ درا

Predicted: 
Actual: میں زیادہ غریب ہیں۔

Predicted: 
Actual: وہ برتن دھو کر فارغ ہوئی۔

Predicted: 
Actual: بچے پارک میں کھیل رہے ہیں۔

Predicted: 
Actual: آسمان پر بادل چھائے ہوئے تھے۔

Predicted:  
Actual: صحیفہ کائنات اور مخلوقِ خدا۔

Predicted: 
Actual: کبھی مت بھولنا

Predicted:  
Actual: بہاولپور

Predicted: 
Actual: میرے دوست

Predicted: 
Actual: دل ہی تو ہے نہ سنگ و خشت

Predicted: 
Actual: یہاں کلک کیجیے۔

Predicted: 
Actual: خوش آمدید۔

Predicted: 
Actual: اب ہمارے دل

In [6]:
save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
image_processor.save_pretrained(save_path)
print(f'Model saved to: {save_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/SI26-urdu-ocr-model
